# 01 — Data Extraction

## Doctor's Consultation Fee Analysis

**Source:** Practo  
**Method:** Web scraping using `Requests` and `BeautifulSoup`

### Objective

This notebook documents the **data extraction stage** of the project and demonstrates the scraping logic used to collect doctor-level consultation information.

The original project was built using specialization-wise scraping notebooks. The final GitHub repository keeps this extraction workflow consolidated into one notebook rather than committing all historical scraping notebooks and intermediate CSV files.

> **Important:** The historical project dataset contains exactly **1,080 records**. This was constructed from **1,066 records in `FinalDataset_Finals.csv` + 14 additional records from `rem_data.csv` = 1,080 records**. The current Practo website may return a different number of records today, so a fresh live scrape is not used to overwrite the project's historical dataset.

## 1. Historical Dataset Construction

The original extraction pipeline produced specialization-level datasets first.

### Specialization-level final files

| Specialization | Records |
|---|---:|
| Cardiologist | 111 |
| Dentist | 120 |
| Dermatologist | 120 |
| ENT | 120 |
| General Physician | 120 |
| General Surgeon | 120 |
| Gynecologist | 120 |
| Neurologist | 115 |
| Orthopedist | 120 |
| **Total** | **1,066** |

These files were combined into:

```text
FinalDataset_Finals.csv
```

with **1,066 records**.

A further 14 records were available in:

```text
rem_data.csv
```

with:

- Cardiologist: 9
- Neurologist: 5

Therefore:

```text
1,066 + 14 = 1,080 records
```

The two datasets were concatenated to create:

```text
Project_DataSets01.csv
```

which contains the project's **1,080-record uncleaned dataset**.

### Historical flow

```text
Specialization-level datasets
            ↓
   FinalDataset_Finals.csv
          1,066
            +
       rem_data.csv
           14
            ↓
   Project_DataSets01.csv
          1,080
            ↓
   Data Cleaning
            ↓
   Project_DataSets02.csv
            ↓
   Feature preparation / renaming
            ↓
   Practo_DataSets.csv
          1,080
            ↓
            EDA
```

The raw and intermediate files above are preserved in the project's development archive rather than committed to the final GitHub repository.

## 2. Libraries

In [2]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np

from time import sleep
from urllib.parse import quote

## 3. Extraction Scope

The original project covered **9 medical specializations across 12 cities**.

The configuration below is used for the reusable live-scraping demonstration. It does **not** imply that a fresh scrape will reproduce exactly 1,080 historical records.

In [3]:
specializations = [
    "Cardiologist",
    "Dentist",
    "Dermatologist",
    "ENT",
    "General Physician",
    "General Surgeon",
    "Gynecologist",
    "Neurologist",
    "Orthopedist",
]

cities = [
    "Hyderabad",
    "Kolkata",
    "Chennai",
    "Delhi",
    "Bangalore",
    "Mumbai",
    "Ahmedabad",
    "Chandigarh",
    "Patna",
    "Lucknow",
    "Pune",
    "Visakhapatnam",
]

base_url = "https://www.practo.com/search/doctors"

print(f"Specializations: {len(specializations)}")
print(f"Cities: {len(cities)}")
print(f"Search combinations: {len(specializations) * len(cities)}")

Specializations: 9
Cities: 12
Search combinations: 108


## 4. Build Practo Search URL

In [4]:
def build_search_url(specialization, city):
    query = (
        '[{"word":"'
        + specialization
        + '","autocompleted":true,"category":"subspeciality"}]'
    )

    return (
        f"{base_url}"
        f"?results_type=doctor"
        f"&q={quote(query)}"
        f"&city={quote(city)}"
    )

# Example URL
build_search_url("General Physician", "Hyderabad")

'https://www.practo.com/search/doctors?results_type=doctor&q=%5B%7B%22word%22%3A%22General%20Physician%22%2C%22autocompleted%22%3Atrue%2C%22category%22%3A%22subspeciality%22%7D%5D&city=Hyderabad'

## 5. Scraping Function

The function below follows the HTML selectors used by the original scraping workflow.

It extracts:

- Doctor name
- Specialization
- Experience
- Hospital / clinic name
- City
- Patient recommendation
- Source
- Consultation fee

A record-count validation is included so that a change in page structure does not silently create a misaligned DataFrame.

In [5]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}


def scrape_practo_page(specialization, city, timeout=20):
    url = build_search_url(specialization, city)

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=timeout
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    doctor_names = [
        x.get_text(strip=True)
        for x in soup.find_all("h2", class_="doctor-name")
    ]

    consultation_fees = [
        x.get_text(strip=True)
        for x in soup.find_all(
            "span",
            attrs={"data-qa-id": "consultation_fee"}
        )
    ]

    hospitals = [
        x.get_text(strip=True)
        for x in soup.find_all(
            "span",
            attrs={"data-qa-id": "doctor_clinic_name"}
        )
    ]

    patient_likes = []
    recommendation_blocks = soup.find_all(
        "div",
        class_="uv2-spacer--md-top uv2-cushion--lg-top "
               "u-border-general-dashed-top--gray-10"
    )

    for block in recommendation_blocks:
        rec = block.find(
            "span",
            attrs={"data-qa-id": "doctor_recommendation"}
        )
        patient_likes.append(
            rec.get_text(strip=True) if rec else None
        )

    scraped_cities = [
        x.get_text(strip=True)
        for x in soup.find_all("span", class_="u-t-capitalize")
    ]

    experiences = []
    experience_blocks = soup.find_all(
        "div",
        attrs={"data-qa-id": "doctor_experience"}
    )

    for block in experience_blocks:
        text = block.get_text(" ", strip=True)
        match = re.search(r"\d+", text)
        experiences.append(int(match.group()) if match else np.nan)

    lengths = {
        "Doctor_Name": len(doctor_names),
        "Experience": len(experiences),
        "Hospital_Name": len(hospitals),
        "City": len(scraped_cities),
        "Patient_Like": len(patient_likes),
        "Consultation_Fee": len(consultation_fees),
    }

    if len(set(lengths.values())) != 1:
        raise ValueError(
            f"Record-count mismatch for {specialization} / {city}: {lengths}"
        )

    return pd.DataFrame({
        "Doctor_Name": doctor_names,
        "Specialization": [specialization] * len(doctor_names),
        "Experience": experiences,
        "Hospital_Name": hospitals,
        "City": scraped_cities,
        "Patient_Like": patient_likes,
        "Source": ["Practo"] * len(doctor_names),
        "Consultation_Fee": consultation_fees,
    })

## 6. Test a Single Extraction

Before attempting any broader collection, test one specialization-city combination.

This confirms whether the current website structure still matches the selectors used in the project.

In [6]:
test_df = scrape_practo_page(
    "General Physician",
    "Hyderabad"
)

display(test_df.head())
print("Shape:", test_df.shape)

,Doctor_Name,Specialization,Experience,Hospital_Name,City,Patient_Like,Source,Consultation_Fee
0,Dr. K K Reddy Sadu,General Physician,25,K K Reddy Hospital,Hyderabad,95%,Practo,₹600
1,Dr. Laxminadh Sivaraju,General Physician,22,Germanten Hospitals,Hyderabad,83%,Practo,₹800
2,Dr. Pradeep Kumar Mishra,General Physician,26,Medicover Hospitals,Hyderabad,93%,Practo,₹1000
3,Dr. Mithil B Ghushe,General Physician,23,Medicover Hospitals - Hitech City,Hyderabad,86%,Practo,₹600
4,Dr. Rahul Gandhi Godisela,General Physician,22,Medicover Hospital,Hyderabad,89%,Practo,₹1000


Shape: (10, 8)


### Expected test behaviour

The exact number of rows can change because Practo's live search results and page structure may change over time.

For the current test used while preparing this project update, the extraction returned **10 records × 8 columns** successfully.

This live result should **not** be used to replace the project's historical 1,080-record dataset.

## 7. Optional Full Live Extraction

The block below is intentionally disabled by default.

A fresh live scrape is exploratory only. It is not the mechanism used to recreate the historical 1,080-record project dataset, because the current website can return a different set and number of doctors.

In [ ]:
RUN_FULL_EXTRACTION = False

all_frames = []

if RUN_FULL_EXTRACTION:
    for specialization in specializations:
        for city in cities:
            print(f"Collecting: {specialization} | {city}")

            try:
                temp_df = scrape_practo_page(
                    specialization=specialization,
                    city=city
                )
                all_frames.append(temp_df)

            except Exception as exc:
                print(f"Skipped {specialization} | {city}: {exc}")

            sleep(1)

    if all_frames:
        live_extracted_df = pd.concat(
            all_frames,
            ignore_index=True
        )

        print("Live extraction shape:", live_extracted_df.shape)
        display(live_extracted_df.head())

    else:
        live_extracted_df = pd.DataFrame()
        print("No data was extracted.")

else:
    live_extracted_df = pd.DataFrame()
    print("Full live extraction is disabled.")

## 8. Extraction Validation

When a live extraction is performed, basic validation should be completed before using the data downstream.

In [ ]:
if not live_extracted_df.empty:
    print("Shape:", live_extracted_df.shape)

    print("\nColumns:")
    print(live_extracted_df.columns.tolist())

    print("\nRecords by specialization:")
    display(
        live_extracted_df["Specialization"]
        .value_counts()
        .sort_index()
    )

    print("\nRecords by city:")
    display(
        live_extracted_df["City"]
        .value_counts()
        .sort_index()
    )

    print("\nDuplicate rows:", live_extracted_df.duplicated().sum())

## 9. Output of the Historical Extraction Stage

The historical extraction stage ultimately produced the 1,080-record dataset that was passed into the cleaning workflow.

The final GitHub repository does **not** commit all raw and intermediate CSV files. Instead, the cleaned project dataset is maintained as:

```text
../data/dataset.csv
```

The next notebook begins from this project dataset and documents the cleaning and preparation steps.

## Next Step

**`02_data_cleaning.ipynb`**

The next stage covers:

- data understanding
- missing-value handling
- percentage-to-numeric conversion
- consultation-fee cleaning
- free-consultation handling
- hospital-name cleanup
- duplicate checks
- outlier detection
- preparation of the analysis-ready dataset